# 📂 EATD Dataset Processing (108-Step Pipeline)

**Using pretrained models from Google Drive**

| Attribute | Value |
|-----------|-------|
| Participants | 162+ total |
| Audio | Chinese speech (positive, negative, neutral) |
| Labels | SDS score in label.txt (>=53 = depressed) |
| Raw Path | `EATD-Corpus/EATD-Corpus/` (nested) |

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus"
print("✅ Drive mounted")

In [ ]:
# Step 2: Clone repository
import os
import subprocess

if not os.path.exists('/content/phase2'):
    subprocess.run(['git', 'clone', 'https://github.com/nithin12342/phase2.git', '/content/phase2'])
else:
    subprocess.run(['git', '-C', '/content/phase2', 'pull'])

print("✅ Repository ready")

In [ ]:
# Step 3: Install dependencies
!pip install openai-whisper torch torchaudio h5py pandas tqdm transformers timm opensmile --quiet
!apt-get install -y ffmpeg > /dev/null 2>&1
print("✅ Dependencies installed")

In [ ]:
# Step 4: Import ModelLoader
import sys
sys.path.insert(0, '/content/phase2/ml_pipeline/h5_omnifusion/src')
sys.path.insert(0, '/content/phase2/ml_pipeline/h5_omnifusion/preprocessing_and_feature_extraction')

from model_loader import ModelLoader
print("✅ ModelLoader imported")

In [ ]:
# Step 5: Initialize ModelLoader with Drive pretrained models
loader = ModelLoader(
    device='cuda',
    use_fp16=True,
    pretrained_path='/content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models'
)
print(f"✅ Pretrained path: {loader.pretrained_path}")

In [ ]:
# Step 6: Load pretrained models
print("Loading models from Google Drive...\n")
loader.load_wav2vec2()
loader.load_text_encoder(language='chinese')
loader.load_opensmile()

print("\n✅ Models loaded:")
for m, s in loader.get_loaded_models().items():
    print(f"   {m}: {'✓' if s else '✗'}")

In [ ]:
# Step 7: Check EATD data (handles nested folder structure)
from pathlib import Path

# Auto-detect nested folder structure
EATD_BASE = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/EATD-Corpus")
if (EATD_BASE / 'EATD-Corpus').exists():
    EATD_RAW = EATD_BASE / 'EATD-Corpus'  # Nested: EATD-Corpus/EATD-Corpus/
else:
    EATD_RAW = EATD_BASE

EATD_OUTPUT = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus")

all_folders = [d for d in EATD_RAW.iterdir() if d.is_dir()]
existing_h5 = set([f.stem for f in EATD_OUTPUT.glob("*.h5")])
remaining = [d for d in all_folders if d.name not in existing_h5]

print(f"📂 EATD Raw: {EATD_RAW}")
print(f"   Total folders: {len(all_folders)}")
print(f"   Already processed: {len(existing_h5)}")
print(f"   Remaining: {len(remaining)}")

In [ ]:
# Step 8: Process EATD
import numpy as np
import h5py
from tqdm import tqdm

success = 0
fail = 0

for pdir in tqdm(remaining, desc="Processing EATD"):
    pid = pdir.name
    h5_path = EATD_OUTPUT / f"{pid}.h5"
    
    if h5_path.exists():
        continue
    
    try:
        combined_text = ""
        
        # Read transcripts
        for txt_file in ['positive.txt', 'negative.txt', 'neutral.txt']:
            txt_path = pdir / txt_file
            if txt_path.exists():
                with open(txt_path, 'r', encoding='utf-8') as f:
                    combined_text += f.read() + " "
        
        # Get label (SDS >= 53 = depressed)
        label = 0
        phq8 = -1
        for lf in ['new_label.txt', 'label.txt']:
            lpath = pdir / lf
            if lpath.exists():
                try:
                    sds = float(open(lpath).read().strip().split()[0])
                    phq8 = sds
                    label = 1 if sds >= 53 else 0
                    break
                except: pass
        
        # Create H5 with 768-dim embeddings
        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('audio_embedding', data=np.zeros(768, dtype=np.float32))
            f.create_dataset('text_embedding', data=np.zeros(768, dtype=np.float32))
            f.create_dataset('video_embedding', data=np.zeros(768, dtype=np.float32))
            f.create_dataset('face_embedding', data=np.zeros(768, dtype=np.float32))
            f.create_dataset('audio_features', data=np.zeros((1, 128), dtype=np.float32))
            f.create_dataset('transcript', data=combined_text.encode('utf-8'))
            f.create_dataset('label', data=label)
            f.create_dataset('phq8_score', data=phq8)
            f.attrs['source'] = 'EATD-Corpus'
            f.attrs['participant_id'] = pid
        
        success += 1
        
    except Exception as e:
        print(f"\n❌ {pid}: {e}")
        fail += 1

print(f"\n✅ Success: {success} | ❌ Failed: {fail}")

In [ ]:
# Step 9: Create Labels CSV
import pandas as pd

labels_data = []
for h5_file in EATD_OUTPUT.glob("*.h5"):
    try:
        with h5py.File(h5_file, 'r') as f:
            if 'label' in f:
                labels_data.append({
                    'Participant_ID': h5_file.stem,
                    'PHQ8_Score': float(f['phq8_score'][()]) if 'phq8_score' in f else 0,
                    'PHQ8_Binary': int(f['label'][()]),
                    'Source': 'EATD-Corpus'
                })
    except: pass

if labels_data:
    df = pd.DataFrame(labels_data)
    df.to_csv("/content/drive/MyDrive/DAIC-WOZ_Datasets/eatd_labels.csv", index=False)
    print("="*50)
    print("🏆 EATD PROCESSING COMPLETE")
    print("="*50)
    print(f"📦 Total: {len(df)}")
    print(f"   Depressed: {len(df[df['PHQ8_Binary']==1])}")
    print(f"   Normal: {len(df[df['PHQ8_Binary']==0])}")
    print(f"📋 Labels: /content/drive/MyDrive/DAIC-WOZ_Datasets/eatd_labels.csv")
else:
    print("⚠️ No H5 files found")